In [1]:
import sys
import os

# Ajoute la racine du projet au path pour pouvoir importer feature_extraction
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print("Project root:", PROJECT_ROOT)
print("sys.path:", sys.path[:3])

Project root: /home/ali/WildFire-Alert/backend
sys.path: ['/home/ali/WildFire-Alert/backend', '/home/ali/miniconda3/envs/wildfire/lib/python311.zip', '/home/ali/miniconda3/envs/wildfire/lib/python3.11']


In [2]:
from automl.feature_extraction import build_datasets, extract_features_from_bytes
print("Import OK")

Import OK


In [ ]:
DATASET_PATH = "/home/ali/WildFire-Alert/backend/data/dataset"

assert os.path.exists(DATASET_PATH), f"Dataset introuvable : {DATASET_PATH}"
print("Dataset trouvé :", DATASET_PATH)

Dataset trouvé : /home/ali/WildFire-Alert/backend/data/dataset


In [ ]:
# Affiche la structure du dataset
for root, dirs, files in os.walk(DATASET_PATH):
    depth = root.replace(DATASET_PATH, '').count(os.sep)
    indent = '  ' * depth
    print(f"{indent}{os.path.basename(root)}/")
    if depth >= 3:  
        subindent = '  ' * (depth + 1)
        print(f"{subindent}[{len(files)} fichiers]")
        dirs.clear()

dataset/
  Archive/
    test/
      nowildfire/
        [5640 fichiers]
      wildfire/
        [6960 fichiers]
    train/
      nowildfire/
        [29000 fichiers]
      wildfire/
        [31500 fichiers]
    valid/
      nowildfire/
        [5640 fichiers]
      wildfire/
        [6960 fichiers]


In [ ]:

print("Démarrage de build_datasets...")
train_df, test_df, dev_df = build_datasets(DATASET_PATH)
print("Extraction terminée !")

Démarrage de build_datasets...
Extraction terminée !


In [11]:
print(f"Train : {train_df.shape}")
print(f"Test  : {test_df.shape}")
print(f"Dev   : {dev_df.shape}")
print()
print("Colonnes :", list(train_df.columns))

Train : (2000, 72)
Test  : (300, 72)
Dev   : (300, 72)

Colonnes : ['exg_mean', 'exg_std', 'exg_min', 'exg_max', 'exg_p25', 'exg_p75', 'exg_skewness', 'exg_kurtosis', 'vci_mean', 'vci_std', 'vci_min', 'vci_max', 'vci_p25', 'vci_p75', 'vci_skewness', 'vci_kurtosis', 'rg_ratio_mean', 'rg_ratio_std', 'rg_ratio_min', 'rg_ratio_max', 'rg_ratio_p25', 'rg_ratio_p75', 'rg_ratio_skewness', 'rg_ratio_kurtosis', 'brightness_mean', 'brightness_std', 'brightness_min', 'brightness_max', 'brightness_p25', 'brightness_p75', 'brightness_skewness', 'brightness_kurtosis', 'saturation_mean', 'saturation_std', 'saturation_min', 'saturation_max', 'saturation_p25', 'saturation_p75', 'saturation_skewness', 'saturation_kurtosis', 'veg_ratio', 'bare_ratio', 'ch_r_mean', 'ch_r_std', 'ch_r_min', 'ch_r_max', 'ch_r_p25', 'ch_r_p75', 'ch_r_skewness', 'ch_r_kurtosis', 'ch_g_mean', 'ch_g_std', 'ch_g_min', 'ch_g_max', 'ch_g_p25', 'ch_g_p75', 'ch_g_skewness', 'ch_g_kurtosis', 'ch_b_mean', 'ch_b_std', 'ch_b_min', 'ch_b_m

In [12]:
train_df.head

<bound method NDFrame.head of       exg_mean   exg_std   exg_min   exg_max   exg_p25   exg_p75  \
0     0.145231  0.093458 -0.050980  0.447059  0.066667  0.227451   
1     0.153387  0.082503 -0.078431  0.364706  0.109804  0.215686   
2     0.134053  0.067519 -0.105882  0.305882  0.117647  0.176471   
3     0.094325  0.052182 -0.074510  0.282353  0.054902  0.125490   
4     0.112418  0.042000 -0.047059  0.235294  0.078431  0.145098   
...        ...       ...       ...       ...       ...       ...   
1995  0.046828  0.055397 -0.690196  0.392157  0.019608  0.058824   
1996  0.130702  0.049360  0.007843  0.309804  0.094118  0.164706   
1997  0.092559  0.059820 -0.196078  0.400000  0.054902  0.129412   
1998  0.074239  0.049980 -0.137255  0.333333  0.039216  0.101961   
1999  0.037997  0.022827 -0.121569  0.196078  0.023529  0.047059   

      exg_skewness  exg_kurtosis  vci_mean   vci_std  ...  ch_b_p25  ch_b_p75  \
0         0.570714     -0.952257  0.085366  0.152932  ...  0.133333  0.3

In [13]:
train_df["target"].value_counts()

target
1    1000
0    1000
Name: count, dtype: int64

In [ ]:

lat, lon = 48.8566, 2.3522
zoom = 15


import math
import requests
def lat_lon_to_tile(lat, lon, zoom):
    n = 2 ** zoom
    x = int((lon + 180) / 360 * n)
    y = int((1 - math.log(math.tan(math.radians(lat)) + 1 / math.cos(math.radians(lat))) / math.pi) / 2 * n)
    return x, y

x_tile, y_tile = lat_lon_to_tile(lat, lon, zoom)

url = f"https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{zoom}/{y_tile}/{x_tile}"

response = requests.get(url)
print("Status:", response.status_code)
print("Taille:", len(response.content))

Status: 200
Taille: 23554


In [6]:
def city_to_coordinates(city_name: str) -> tuple:
    """
    Convertit un nom de ville en (lat, lon) via Nominatim (OpenStreetMap)
    """
    url = "https://nominatim.openstreetmap.org/search"
    params = {
        "q": city_name,
        "format": "json",
        "limit": 1
    }
    headers = {"User-Agent": "WildFireAlert/1.0"}  # obligatoire pour Nominatim

    response = requests.get(url, params=params, headers=headers)
    results = response.json()

    if not results:
        raise ValueError(f"Ville introuvable : {city_name}")

    lat = float(results[0]["lat"])
    lon = float(results[0]["lon"])

    return lat, lon

def lat_lon_to_tile(lat, lon, zoom):
    n = 2 ** zoom
    x = int((lon + 180) / 360 * n)
    y = int((1 - math.log(math.tan(math.radians(lat)) + 1 / math.cos(math.radians(lat))) / math.pi) / 2 * n)
    return x, y

In [ ]:
import requests, math

city = "Marseille"

lat, lon = city_to_coordinates(city)

x_tile, y_tile = lat_lon_to_tile(lat, lon, zoom=15)
url = f"https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/15/{y_tile}/{x_tile}"
response = requests.get(url)
print("Status:", response.status_code)
print("Taille:", len(response.content))

Status: 200
Taille: 25333


In [9]:
x = extract_features_from_bytes(response.content)

In [10]:
print(x)

   feature_1  feature_2  feature_3  feature_4  feature_5  feature_6  \
0   0.055125   0.049594  -0.215686   0.345098   0.031373   0.082353   

   feature_7  feature_8  feature_9  feature_10  ...  feature_62  feature_63  \
0  -0.501062   1.649125   0.056012    0.191294  ...         1.0    0.117647   

   feature_64  feature_65  feature_66  feature_67  feature_68  feature_69  \
0    0.541176    0.205154   -1.060922  131.628904    0.002087    0.276554   

   feature_70  feature_71  
0    0.756262    0.304764  

[1 rows x 71 columns]
